In [ ]:
import pandas as pd
import json
from tqdm.notebook import tqdm

# Decompress ClassyFire

In [ ]:
import zstandard as zstd

input_path = 'data/classyfire.v4.tsv.zst' # -> 2.1 GB
output_path = 'data/classyfire.v4.tsv'    # -> ~24 GB

file_size = __import__('os').path.getsize(input_path)
    
decomp = zstd.ZstdDecompressor()

with open(input_path, 'rb') as f_in, \
        open(output_path, 'wb') as f_out, \
        decomp.stream_reader(f_in) as reader, \
        tqdm(total=file_size, unit='B', unit_scale=True, desc="Decompressing") as pbar:
    
    while True:
        chunk = reader.read(1024*1024)
        while chunk:
            f_out.write(chunk)
            pbar.update(len(chunk))

print("Done!")

# MassSpecGym dataset

## Overview

In [6]:
msg_df = pd.read_csv('data/MassSpecGym1.5.tsv', sep='\t')
print(msg_df.shape)
msg_df.info()

(231104, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231104 entries, 0 to 231103
Data columns (total 14 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   identifier            231104 non-null  object 
 1   mzs                   231104 non-null  object 
 2   intensities           231104 non-null  object 
 3   smiles                231104 non-null  object 
 4   inchikey              231104 non-null  object 
 5   formula               231104 non-null  object 
 6   precursor_formula     231104 non-null  object 
 7   parent_mass           231104 non-null  float64
 8   precursor_mz          231104 non-null  float64
 9   adduct                231104 non-null  object 
 10  instrument_type       225881 non-null  object 
 11  collision_energy      121746 non-null  float64
 12  fold                  231104 non-null  object 
 13  simulation_challenge  231104 non-null  bool   
dtypes: bool(1), float64(3), object(10)
memo

In [7]:
msg_df['fold'].value_counts()

fold
train    194119
val       19429
test      17556
Name: count, dtype: int64

## Processing

In [55]:
test_msg_df = msg_df[msg_df['fold'] == 'test']
print(test_msg_df.shape)
test_msg_df.info()

(17556, 14)
<class 'pandas.core.frame.DataFrame'>
Index: 17556 entries, 168 to 231100
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            17556 non-null  object 
 1   mzs                   17556 non-null  object 
 2   intensities           17556 non-null  object 
 3   smiles                17556 non-null  object 
 4   inchikey              17556 non-null  object 
 5   formula               17556 non-null  object 
 6   precursor_formula     17556 non-null  object 
 7   parent_mass           17556 non-null  float64
 8   precursor_mz          17556 non-null  float64
 9   adduct                17556 non-null  object 
 10  instrument_type       17147 non-null  object 
 11  collision_energy      10162 non-null  float64
 12  fold                  17556 non-null  object 
 13  simulation_challenge  17556 non-null  bool   
dtypes: bool(1), float64(3), object(10)
memory usage: 1.9+ MB


In [ ]:
test_msg_df['inchikey'].value_counts()

inchikey
RWXIFXNRCLMQCD    383
QOVKKEKBURQILF    194
NWXMGUDVXFXRIG    159
RXZBMPWDPOLZGW    139
ZVTVWDXRNMHGNY    126
                 ... 
LBEJUQWGEGOTNX      1
QGVLYPPODPLXMB      1
WEGITEKALFVLHV      1
KVHRYLNQDWXAGI      1
HVKUYPXKTAMIFI      1
Name: count, Length: 2998, dtype: int64

In [56]:
dedupl_msg_df = test_msg_df.drop_duplicates('inchikey')
print(dedupl_msg_df.shape)
dedupl_msg_df.info()

(2998, 14)
<class 'pandas.core.frame.DataFrame'>
Index: 2998 entries, 168 to 222364
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   identifier            2998 non-null   object 
 1   mzs                   2998 non-null   object 
 2   intensities           2998 non-null   object 
 3   smiles                2998 non-null   object 
 4   inchikey              2998 non-null   object 
 5   formula               2998 non-null   object 
 6   precursor_formula     2998 non-null   object 
 7   parent_mass           2998 non-null   float64
 8   precursor_mz          2998 non-null   float64
 9   adduct                2998 non-null   object 
 10  instrument_type       2753 non-null   object 
 11  collision_energy      2269 non-null   float64
 12  fold                  2998 non-null   object 
 13  simulation_challenge  2998 non-null   bool   
dtypes: bool(1), float64(3), object(10)
memory usage: 330.8+ KB


# Load ClassyFire

In [121]:
main_df = test_msg_df.copy()
main_df['chemont_tree'] = None

classyfire_dfs = pd.read_csv('data/classyfire.v4.tsv', sep='\t', chunksize=2048)

for df in tqdm(classyfire_dfs):
    df = classyfire_dfs.get_chunk(2048)
    df['inchikey'] = df['inchikey'].apply(lambda key: key.split("-")[0])  # Keep only first block of the Inchikey
    df.drop_duplicates('inchikey', inplace=True)
    df = df[['inchikey', 'chemont_tree_json']]

    main_df = pd.merge(main_df, df, on='inchikey', how='outer')

    main_df['chemont_tree'].where(main_df['chemont_tree'].notnull(), main_df['chemont_tree_json'], inplace=True)

    main_df.drop('chemont_tree_json', axis=1, inplace=True)

main_df.info()
main_df.to_csv("data/MSG_testset_enriched.tsv", sep='\t')

1361it [42:04,  1.85s/it]


KeyboardInterrupt: 

# Enrich MassSepcGym dataset with ClassyFire